In [1]:
import numpy as np 
import matplotlib.pylab as plt
import os 
import json 
import pandas as pd 
import random

save_dir = "./resources/"

stim_file = f"{save_dir}/cogsci_final_stimuli.csv"#/stim_design.csv"

In [2]:
all_stim_data = pd.read_csv(stim_file)
print("Num stim: ", len(all_stim_data))
all_stim_data.head(3)

Num stim:  121


,Board size,Winning condition,Label,Special features
0,3 x 3,3 pieces in a row wins.,simple-win,NaN
1,4 x 4,3 pieces in a row wins.,simple-win,NaN
2,5 x 5,3 pieces in a row wins.,simple-win,NaN


In [3]:
baseline_level = all_stim_data.iloc[0]
stim_data = all_stim_data[1:].reset_index()
len(stim_data)

120

In [4]:
len(stim_data)/10

12.0

In [5]:
np.random.seed(7)
random.seed(7)
n_stim = np.arange(len(stim_data))
np.random.shuffle(n_stim)
all_batch_idxs = np.array_split(n_stim, 12) 

In [6]:
len(all_batch_idxs[0]), len(all_batch_idxs)

(10, 12)

In [7]:
# set up stimuli

tasks = ["how_fun", "advantage"]


# each participant sees all win lengths for a particular task, shuffled 

all_batch_data = {}

current_cond_num = 0
for task_idx, task in enumerate(tasks): 
    print(f"Batches for {task} start at {current_cond_num}")
    for batch_idxs in all_batch_idxs: 
        batch_data = []
        batch = stim_data.loc[batch_idxs]
        # concatenate tic-tac-toe for all
        #batch_stims = pd.concat([baseline_level, batch])
        batch_stims = batch.append(baseline_level, ignore_index = True)
        for _, entry in batch_stims.iterrows(): 
            board_size = entry["Board size"]
            win_conds = entry["Winning condition"]
            
            board_rows, board_cols = board_size.split(" x ")
            #game_img_pth = draw_go_board_with_dot_and_color(board_rows,
#                                                             board_cols)
            if board_rows == "Inf": board_rows = "Infinity"
            if board_cols == "Inf": board_cols = "Infinity"
            
            stimuli = {
                "task": task,
                "board_cols": board_cols,
                "board_rows": board_rows,
                "stim_tag": entry["Label"],
                "win_conditions": win_conds,
            }
            batch_data.append(stimuli)
        all_batch_data[current_cond_num] = batch_data
        current_cond_num+=1




Batches for how_fun start at 0
Batches for advantage start at 12


In [8]:
# check to make sure all games are seen twice (once per task)
game_counts = {}
for batch_cond, batch_data in all_batch_data.items(): 
    for game in batch_data: 
        game_uid = f"{game['board_cols']}*{game['board_rows']}*{game['win_conditions']}" 
        if game_uid not in game_counts: game_counts[game_uid] = 1
        else: game_counts[game_uid] += 1
print(len(game_counts.keys()))
for k,v in game_counts.items():
    if v != 2: print(k,v)

121
3*3*3 pieces in a row wins. 24


In [9]:
# write out directly as a js file 
# help from: https://stackoverflow.com/questions/32284317/send-python-information-to-a-javascript-file
with open(f'{save_dir}batches.js', 'w') as f:
    f.write('var batch_data = %s;' % json.dumps(all_batch_data))

In [10]:
len(all_batch_data)

24

In [13]:
all_batch_data[13]

[{'task': 'advantage',
  'board_cols': '10',
  'board_rows': '10',
  'stim_tag': 'player1-constraintB',
  'win_conditions': 'Each player needs 3 pieces in a row to win. The first player can only win by making a diagonal row, but the second player does not have this restriction.'},
 {'task': 'advantage',
  'board_cols': '5',
  'board_rows': '5',
  'stim_tag': 'only-diag',
  'win_conditions': '3 pieces in a row wins. However, a player can only win by making a diagonal row. Horizontal and vertical rows do not count.'},
 {'task': 'advantage',
  'board_cols': '4',
  'board_rows': '4',
  'stim_tag': 'loss',
  'win_conditions': '3 pieces in a row loses.'},
 {'task': 'advantage',
  'board_cols': '10',
  'board_rows': '10',
  'stim_tag': 'player1-2pieces',
  'win_conditions': '4 pieces in a row wins. The first player can place 2 pieces as their first move, while the second player can only place 1 piece as their first move.'},
 {'task': 'advantage',
  'board_cols': '10',
  'board_rows': '1',
  '